In [ ]:
# Clear environment
rm(list = ls())

if(!require(dplyr)) install.packages("dplyr")
if(!require(randomForest)) install.packages("randomForest")
if(!require(gbm)) install.packages("gbm")
if(!require(gam)) install.packages("gam")
if(!require(caret)) install.packages("caret")
if(!require(e1071)) install.packages("e1071")
if(!require(ROCR)) install.packages("ROCR")
if(!require(pROC)) install.packages("pROC")
if(!require(themis)) install.packages("themis") 
if(!require(patchwork)) install.packages("patchwork")
# if (!requireNamespace("ROCR", quietly = TRUE)) {install.packages("ROCR")}

update.packages(c("themis", "recipes", "gbm", "randomForest", "gam", "caret", "e1071", "ROCR"))

library('dplyr');
library("pROC")

#Read the data
## please change the file location
setwd('~/Cathy/OMS/Data & Visual Analytics - CSE6242OAN O01 O3 AO/Project/data/cleaned_data');

Raw_Data <- read.csv(file = "cleaned_data_updated.csv", header=TRUE, dec=",");

# Check unique value of contributing factors
# lapply(Raw_Data[c('CONTRIBUTING_FACTOR_VEHICLE_1', 'CONTRIBUTING_FACTOR_VEHICLE_2')], unique); 

# Function to group contributing factors 
group_factors <- function(factors) {
  factor_group <- sapply(factors, function(factor) {
    if (factor %in% c("Driver Inexperience", "Driver Inattention/Distraction", "Aggressive Driving/Road Rage", 
                      "Texting", "Cell Phone (hands-free)", "Eating or Drinking", "Passenger Distraction", "Fell Asleep", 
                      "Backing Unsafely", 
                      "Cell Phone (hand-Held)", "Using On Board Navigation Device", 
                      "Other Electronic Device", "Listening/Using Headphones", 
                      "Cell Phone (hand-held)")) {
      return("Driver Behavior/Inattention")
    } else if (factor %in% c("Brakes Defective", "Tire Failure/Inadequate", "Steering Failure", "Headlights Defective", 
                             "Windshield Inadequate", "Other Lighting Defects", "Tow Hitch Defective", "Oversized Vehicle", "Accelerator Defective", "Driverless/Runaway Vehicle", 
                             "Vehicle Vandalism", "Tinted Windows")) {
      return("Vehicle Condition/Mechanical Failure")
    } else if (factor %in% c("Traffic Control Disregarded", "Lane Marking Improper/Inadequate", "Traffic Control Device Improper/Non-Working")) {
      return("Traffic Control Issues")
    } else if (factor %in% c("Pavement Slippery", "Pavement Defective", "Shoulders Defective/Improper", "Obstruction/Debris")) {
      return("Road Conditions")
    } else if (factor %in% c("Glare", "View Obstructed/Limited")) {
      return("Environmental Factors")
    } else if (factor %in% c("Unsafe Speed", "Following Too Closely", "Passing Too Closely", "Passing or Lane Usage Improper", 
                             "Unsafe Lane Changing", "Failure to Yield Right-of-Way", "Failure to Keep Right", "Turning Improperly", "Oversized Vehicle", 
                             "Alcohol Involvement", "Drugs (illegal)", "Drugs (Illegal)", "Prescription Medication")) {
      return("Traffic Violations/Unsafe Driving")
    } else if (factor %in% c("Pedestrian/Bicyclist/Other Pedestrian Error/Confusion", "Reaction to Uninvolved Vehicle", 
                             "Reaction to Other Uninvolved Vehicle", "Animals Action", "Other Vehicular", "Outside Car Distraction")) {
      return("Pedestrian/Other Vehicle Involvement")
    } else if (factor %in% c("Unspecified", "MISSING", "80", "1")) {
      return("Unspecified or Missing")
    } else if (factor %in% c("Illness", "Illnes", "Lost Consciousness", "Physical Disability", "Fatigued/Drowsy")) {
      return("Health Issues")
    } else {
      return("Other")
    }
  })
  
  return(factor_group)
}

# # Filter rows where 'GROUPED_FACTOR_VEHICLE_1' is categorized as 'Other'
# other_vehicle_1 <- Raw_Data[Raw_Data$GROUPED_FACTOR_VEHICLE_1 == "Other", ]
# unique_values_vehicle_1 <- unique(other_vehicle_1$CONTRIBUTING_FACTOR_VEHICLE_1)
# 
# other_vehicle_2 <- Raw_Data[Raw_Data$GROUPED_FACTOR_VEHICLE_2 == "Other", ]
# unique_values_vehicle_2 <- unique(other_vehicle_2$CONTRIBUTING_FACTOR_VEHICLE_2)
# 
# unique_others <- unique(c(unique_values_vehicle_1, unique_values_vehicle_2))
# print(unique_others)

Raw_Data$GROUPED_FACTOR_VEHICLE_1 <- group_factors(Raw_Data$CONTRIBUTING_FACTOR_VEHICLE_1)
Raw_Data$GROUPED_FACTOR_VEHICLE_2 <- group_factors(Raw_Data$CONTRIBUTING_FACTOR_VEHICLE_2)

unique_grouped_factor_count <- length(unique(c(Raw_Data$GROUPED_FACTOR_VEHICLE_1, Raw_Data$GROUPED_FACTOR_VEHICLE_2)))
print(unique_grouped_factor_count) 

# lapply(Raw_Data[c('GROUPED_FACTOR_VEHICLE_1', 'GROUPED_FACTOR_VEHICLE_2')], unique); 

# Raw_Data %>%
#   group_by(GROUPED_FACTOR_VEHICLE_1) %>%
#   summarise(count = n());
# 
# Raw_Data %>%
#   group_by(GROUPED_FACTOR_VEHICLE_2) %>%
#   summarise(count = n());

# Convert data types

# Check if the "Data" data frame exists
if (exists("Data")) {
  # If it exists, remove it
  rm(Data)
  print("Data data frame dropped.")
} else {
  print("Data data frame does not exist.")
}


Data <- Raw_Data %>%
  mutate(
    NUMBER_OF_PERSONS_INJURED = as.numeric(NUMBER_OF_PERSONS_INJURED),
    NUMBER_OF_PERSONS_KILLED = as.numeric(NUMBER_OF_PERSONS_KILLED),
    LATITUDE_CRASH = as.numeric(LATITUDE_CRASH),
    LONGITUDE_CRASH = as.numeric(LONGITUDE_CRASH),
    PRCP = as.numeric(PRCP),
    SNOW = as.numeric(SNOW),
    SNWD = as.numeric(SNWD),
    TMAX = as.numeric(TMAX),
    TMIN = as.numeric(TMIN),
    BOROUGH = as.numeric(factor(BOROUGH)),  # Convert categorical to factor
    ZIP_CODE = as.numeric(factor(ZIP_CODE)),
    GROUPED_FACTOR_VEHICLE_1 = as.numeric(factor(GROUPED_FACTOR_VEHICLE_1)),
    GROUPED_FACTOR_VEHICLE_2 = as.numeric(factor(GROUPED_FACTOR_VEHICLE_2)),
    VEHICLE_1_TYPE_REGROUP = as.numeric(factor(VEHICLE_TYPE_CODE_1_REGROUP)),
    VEHICLE_2_TYPE_REGROUP = as.numeric(factor(VEHICLE_TYPE_CODE_2_REGROUP)),
    RUSH_HOUR = as.numeric(factor(RUSH_HOUR)),
    SEASON = as.numeric((factor(SEASON)))
  ) %>%
  select(-CONTRIBUTING_FACTOR_VEHICLE_1, -CONTRIBUTING_FACTOR_VEHICLE_2, -VEHICLE_TYPE_CODE_1_REGROUP, -VEHICLE_TYPE_CODE_2_REGROUP) 


# Create a dummy variable
Data$DUMMY_INJURED <- ifelse(Data$NUMBER_OF_PERSONS_INJURED > 0, 1, 0);
Data$DUMMY_KILLED <- ifelse(Data$NUMBER_OF_PERSONS_KILLED > 0, 1, 0);
Data <- Data %>% select(DUMMY_KILLED, DUMMY_INJURED, everything());
# head(Data);

# check unique value of dummy variables and their distribution
lapply(Data[c('DUMMY_KILLED', 'DUMMY_INJURED')], unique); 


Data %>%
  group_by(DUMMY_KILLED) %>%
  summarise(count = n());
# DUMMY_KILLED   count
# <dbl>   <int>
#  0 1648023
#  1    2425


Data %>%
  group_by(DUMMY_INJURED) %>%
  summarise(count = n());
# DUMMY_INJURED   count
# <dbl>   <int>
#  0 1247274
#  1  403174

Cor_Data <- Data %>%
  select(-c('ON_STREET_NAME',
            'CROSS_STREET_NAME',
            'CRASH_DATETIME',
            'NUMBER_OF_PEDESTRIANS_INJURED', 
            'NUMBER_OF_PEDESTRIANS_KILLED',
            'NUMBER_OF_PERSONS_INJURED', 
            'NUMBER_OF_PERSONS_KILLED',
            'NUMBER_OF_CYCLIST_INJURED',
            'NUMBER_OF_CYCLIST_KILLED',
            'NUMBER_OF_MOTORIST_INJURED',
            'NUMBER_OF_MOTORIST_KILLED',
            'VEHICLE_TYPE_CODE_1',
            'VEHICLE_TYPE_CODE_2',
            'COLLISION_ID', 
            'STATION',
            'LATITUDE_WEATHER_STATION',
            'LONGITUDE_WEATHER_STATION',
            'DISTANCE_FROM_WEATHER_STATION',
            'NAME',                     
            'ELEVATION'))

dim(Cor_Data); 
# 1650448      22

# OR read the Cor_Data directly from another file
# Cor_Data <- read.csv(file = "cleaned_data_final.csv", header=TRUE, dec=",");

# Cor_Data <- Cor_Data %>%
#   mutate(
#     LATITUDE_CRASH = as.numeric(LATITUDE_CRASH),
#     LONGITUDE_CRASH = as.numeric(LONGITUDE_CRASH),
#     PRCP = as.numeric(PRCP),
#     SNOW = as.numeric(SNOW),
#     SNWD = as.numeric(SNWD),
#     TMAX = as.numeric(TMAX),
#     TMIN = as.numeric(TMIN)
#   ) 

dim(Cor_Data); 
# 1650448      22
head(Cor_Data); 

Cor_Data %>%
  group_by(DUMMY_KILLED) %>%
  summarise (Count=n())
# # A tibble: 2 × 2
# DUMMY_KILLED   Count
# <dbl>   <int>
# 0 1648023
# 1    2425

2425/(2425+1648023) # [1] 0.001469298

summary(Cor_Data)


########################EDA######################################################
#-------------------------------------------------------------------------------#
# Scatter_Matrix

corr_data = cor(Cor_Data);
corr_data;
library(corrplot);
plot.new()
dev.off()
corrplot(corr_data, method="color", tl.cex = 0.5, cl.cex = 0.5);
corrplot(corr_data, method="number", tl.cex = 0.5, number.cex = 0.5, cl.cex = 0.5);
corrplot(corr_data, method="ellipse",tl.cex = 0.5, cl.cex = 0.5); 

# corrplot for dummy_killed vs. predictors
cor_data_Road = cor(Data %>% select(c('DUMMY_KILLED','RUSH_HOUR','BOROUGH', 'ZIP_CODE')));
corrplot(cor_data_Road, method="color", tl.cex = 0.5, cl.cex = 0.5);
corrplot(cor_data_Road, method="number", tl.cex = 0.5, number.cex = 0.5, cl.cex = 0.5);

cor_data_VEHICLE = cor(Data %>% select(c('DUMMY_KILLED','RUSH_HOUR','VEHICLE_TYPE_CODE_1_REGROUP','VEHICLE_TYPE_CODE_1_REGROUP','BOROUGH')));
corrplot(cor_data_VEHICLE, method="color", tl.cex = 0.5, cl.cex = 0.5);
corrplot(cor_data_VEHICLE, method="number", tl.cex = 0.5, number.cex = 0.5, cl.cex = 0.5);

cor_data_Weather = cor(Data %>% select(c('DUMMY_KILLED','RUSH_HOUR','SEASON','PRCP', 'SNOW', 'SNWD', 'TMAX', 'TMIN')));
corrplot(cor_data_Weather, method="color", tl.cex = 0.5, cl.cex = 0.5);
corrplot(cor_data_Weather, method="number", tl.cex = 0.5, number.cex = 0.5, cl.cex = 0.5);

cor_data_Road_Weather = cor(Data %>% select(c('DUMMY_KILLED','RUSH_HOUR','SEASON','VEHICLE_TYPE_CODE_1_REGROUP','VEHICLE_TYPE_CODE_1_REGROUP','BOROUGH', 'ZIP_CODE','PRCP', 'SNOW', 'SNWD', 'TMAX', 'TMIN')));
corrplot(cor_data_Road_Weather, method="color", tl.cex = 0.5, cl.cex = 0.5);
corrplot(cor_data_Road_Weather, method="number", tl.cex = 0.5, number.cex = 0.5, cl.cex = 0.5);

# corrplot for dummy_injured vs. predictors
cor_data_Road2 = cor(Data %>% select(c('DUMMY_INJURED','RUSH_HOUR','BOROUGH', 'ZIP_CODE')));
corrplot(cor_data_Road2, method="color", tl.cex = 0.5, cl.cex = 0.5);
corrplot(cor_data_Road2, method="number", tl.cex = 0.5, number.cex = 0.5, cl.cex = 0.5);

cor_data_VEHICLE2 = cor(Data %>% select(c('DUMMY_INJURED','RUSH_HOUR','VEHICLE_TYPE_CODE_1_REGROUP','VEHICLE_TYPE_CODE_1_REGROUP','BOROUGH')));
corrplot(cor_data_VEHICLE2, method="color", tl.cex = 0.5, cl.cex = 0.5);
corrplot(cor_data_VEHICLE2, method="number", tl.cex = 0.5, number.cex = 0.5, cl.cex = 0.5);

cor_data_Weather2 = cor(Data %>% select(c('DUMMY_INJURED','RUSH_HOUR','SEASON','PRCP', 'SNOW', 'SNWD', 'TMAX', 'TMIN')));
corrplot(cor_data_Weather2, method="color", tl.cex = 0.5, cl.cex = 0.5);
corrplot(cor_data_Weather2, method="number", tl.cex = 0.5, number.cex = 0.5, cl.cex = 0.5);

cor_data_Road_Weather2 = cor(Data %>% select(c('DUMMY_INJURED','RUSH_HOUR','SEASON','VEHICLE_TYPE_CODE_1_REGROUP','VEHICLE_TYPE_CODE_1_REGROUP','BOROUGH', 'ZIP_CODE','PRCP', 'SNOW', 'SNWD', 'TMAX', 'TMIN')));
corrplot(cor_data_Road_Weather2, method="color", tl.cex = 0.5, cl.cex = 0.5);
corrplot(cor_data_Road_Weather2, method="number", tl.cex = 0.5, number.cex = 0.5, cl.cex = 0.5);

# boxplots
# head(Data)
library(ggplot2)
library(GGally)

font_size <- 10 # Set your desired font size

for (i in 2:dim(Cor_Data)[2]) {
  print(
    ggplot(Cor_Data, aes(x = as.factor(DUMMY_KILLED), y = Cor_Data[, i])) +
      geom_boxplot() +
      labs(y = colnames(Cor_Data)[i]) +
      theme(
        axis.title = element_text(size = font_size),
        axis.text = element_text(size = font_size),
        plot.title = element_text(size = font_size),
        text = element_text(size = font_size)
      )
  )
}
# According to boxplots, the following predictors have different distribution for dummy_killed 1 vs. 0: 
# significant difference: BOROUGH, RUSH_HOUR, YEAR, VEHICLE_2_TYPE_REGROUP, HOUR
# Moderate difference: LATITUDE_CRASH, GROUPED_FACTOR_VEHICLE_1, SNOW, SNWD, PCRP, TMIN, HOUR
# Minimum difference: ZIP_CODE, lONGTITUDE_CRASH, TMAX, MONTH, DAY, VEHICLE_1_TYPE_REGROUP, SEASON, GROUPED_FACTOR_VEHICLE_2, WEEKDAY

##-------------------------------------------------------------------------##
######################END of EDA#############################################

##############Prepare datasets for modeling##################################
#---------------------------------------------------------------------------#

# Load necessary libraries
library(smotefamily)
library(ROCR)
library(caret)  # For confusionMatrix

# Define helper functions for repeated operations
get_sample_data <- function(data, prop = 0.10, seed = 123) {
  set.seed(seed)
  n_rows <- nrow(data)
  random_indices <- sample(1:n_rows, floor(prop * n_rows), replace = FALSE)
  return(data[random_indices, ])
}

# Clean and preprocess the data
Data1 <- Cor_Data
dim(Data1)  # [1] 1650448 22

# Subset the data, remove multicollinear or irrelevant columns
Data2 <- subset(Cor_Data, select = c(DUMMY_KILLED, DUMMY_INJURED, BOROUGH, RUSH_HOUR, YEAR,
                                     VEHICLE_1_TYPE_REGROUP, VEHICLE_2_TYPE_REGROUP,
                                     LATITUDE_CRASH, GROUPED_FACTOR_VEHICLE_1, SNOW, SNWD, PRCP, TMIN))
dim(Data2)  # [1] 1650448 13

# -------------------- Split Data into Training and Testing Sets --------------------
set.seed(6242)
n <- nrow(Data1)
n1 <- round(n * 0.20)  # 20% for testing

flag <- sort(sample(1:n, n1))
Data1_train <- Data1[-flag, ]
Data1_test <- Data1[flag, ]
Data2_train <- Data2[-flag, ]
Data2_test <- Data2[flag, ]

# Sample 10% of data for smaller subset processing
Data1_train_10 <- get_sample_data(Data1_train)
Data1_test_10 <- get_sample_data(Data1_test)
Data2_train_10 <- get_sample_data(Data2_train)
Data2_test_10 <- get_sample_data(Data2_test)

# Remove the highly correlated column (column 2 DUMMY_INJURED)
remove_column <- function(data) {
  data[, -2]
}

Data1_train_processed <- remove_column(Data1_train)
Data1_test_processed <- remove_column(Data1_test)
Data1_train_10_processed <- remove_column(Data1_train_10)
Data1_test_10_processed <- remove_column(Data1_test_10)
Data2_train_processed <- remove_column(Data2_train)
Data2_test_processed <- remove_column(Data2_test)
Data2_train_10_processed <- remove_column(Data2_train_10)
Data2_test_10_processed <- remove_column(Data2_test_10)

# Extract true response values
y1 <- Data1_train$DUMMY_KILLED
y2 <- Data1_test$DUMMY_KILLED
y3 <- Data2_train$DUMMY_KILLED
y4 <- Data2_test$DUMMY_KILLED

y1_10 <- Data1_train_10$DUMMY_KILLED
y2_10 <- Data1_test_10$DUMMY_KILLED
y3_10 <- Data2_train_10$DUMMY_KILLED
y4_10 <- Data2_test_10$DUMMY_KILLED

summary(Data1_train_processed)

# -------------------- Function to Calculate F1 Score --------------------
F1_Score <- function(predicted, actual) {
  tp <- sum(predicted == 1 & actual == 1)
  fp <- sum(predicted == 1 & actual == 0)
  fn <- sum(predicted == 0 & actual == 1)
  
  # Check for zero division errors and handle them
  precision <- ifelse((tp + fp) == 0, 0, tp / (tp + fp))
  recall <- ifelse((tp + fn) == 0, 0, tp / (tp + fn))
  
  # Handle edge case where precision and recall are both zero
  f1 <- ifelse((precision + recall) == 0, 0, 2 * precision * recall / (precision + recall))
  
  return(f1)
}

# -------------------- Logistic Regression with SMOTE --------------------
library(caret)
library(pROC)

# Cutoff values to evaluate
cutoff_values <- seq(0.1, 0.9, 0.1)

logistic_regression_smote <- function(train_data, test_data, response_col = "DUMMY_KILLED") {
  # Separate features and target
  X_train <- train_data[, -which(names(train_data) == response_col)]
  y_train <- train_data[[response_col]]
  
  # Apply SMOTE
  smote_output <- SMOTE(X_train, y_train, K = 5)
  train_smote <- smote_output$data
  colnames(train_smote)[ncol(train_smote)] <- response_col
  train_smote[[response_col]] <- as.numeric(as.character(train_smote[[response_col]]))
  
  # Logistic Regression Model
  mod_glm_test <- glm(DUMMY_KILLED ~ ., family = binomial, data = train_smote, 
                      weights = ifelse(train_smote$DUMMY_KILLED == 1, 1, 10))
  
  Train_Error_glm <- NULL
  Test_Error_glm <- NULL
  F1_Score_train <- NULL
  F1_Score_test <- NULL
  AUC_train <- NULL
  AUC_test <- NULL
  
  # Calculate Train and Test Errors for different cutoffs
  for (c in cutoff_values) {
    # Predictions on training data and test data
    y1hat <- ifelse(predict(mod_glm_test, train_smote[, -which(names(train_smote) == response_col)], type = "response") < c, 0, 1)
    y2hat <- ifelse(predict(mod_glm_test, test_data[, -which(names(test_data) == response_col)], type = "response") < c, 0, 1)
    
    # Train and Test Errors
    Train_Error_glm <- c(Train_Error_glm, mean(y1hat != train_smote$DUMMY_KILLED))
    Test_Error_glm <- c(Test_Error_glm, mean(y2hat != test_data[[response_col]]))
    
    # Calculate F1 Score for training and testing
    F1_train <- F1_Score(train_smote$DUMMY_KILLED, y1hat)
    F1_test <- F1_Score(test_data[[response_col]], y2hat)
    F1_Score_train <- c(F1_Score_train, F1_train)
    F1_Score_test <- c(F1_Score_test, F1_test)
    
    # Calculate AUC for training and testing
    train_probs <- predict(mod_glm_test, newdata = train_smote[, -which(names(train_smote) == response_col)], type = "response")
    test_probs <- predict(mod_glm_test, newdata = test_data[, -which(names(test_data) == response_col)], type = "response")
    
    AUC_train <- c(AUC_train, auc(roc(train_smote$DUMMY_KILLED, train_probs)))
    AUC_test <- c(AUC_test, auc(roc(test_data[[response_col]], test_probs)))
  }
  
  # Return all metrics as a list
  return(list(Train_Error_glm = Train_Error_glm, 
              Test_Error_glm = Test_Error_glm,
              F1_Score_train = F1_Score_train,
              F1_Score_test = F1_Score_test,
              AUC_train = AUC_train,
              AUC_test = AUC_test))
}

# Apply Logistic Regression with SMOTE for Data1
glm1 <- logistic_regression_smote(Data1_train_processed, Data1_test_processed, response_col = "DUMMY_KILLED")
print(glm1)
# $Train_Error_glm
# [1] 0.3488059 0.4505825 0.4961929 0.5000137 0.4999549 0.4999549 0.4999549 0.4999549 0.4999549
# 
# $Test_Error_glm
# [1] 0.333872580 0.064170378 0.009103578 0.001641976 0.001499591 0.001499591 0.001499591
# [8] 0.001499591 0.001499591
# 
# $F1_Score_train
# [1] 0.64579246 0.26464897 0.02957262 0.00000000 0.00000000 0.00000000 0.00000000 0.00000000
# [9] 0.00000000
# 
# $F1_Score_test
# [1] 0.005289095 0.008333333 0.008578027 0.000000000 0.000000000 0.000000000 0.000000000
# [8] 0.000000000 0.000000000
# 
# $AUC_train
# [1] 0.7154519 0.7154519 0.7154519 0.7154519 0.7154519 0.7154519 0.7154519 0.7154519 0.7154519
# 
# $AUC_test
# [1] 0.680541 0.680541 0.680541 0.680541 0.680541 0.680541 0.680541 0.680541 0.680541

# Plot and evaluate errors
Test_Error_glm1 = glm1$Test_Error_glm
F1_Score_test1 = glm1$F1_Score_test
AUC_test1 = glm1$AUC_test

plot(cutoff_values, Test_Error_glm1)

plot(cutoff_values, Test_Error_glm1, type = "b", col = "blue", pch = 16, xlab = "Cutoff", ylab = "Error")
lines(cutoff_values, F1_Score_test1, col = "red", pch = 17)
lines(cutoff_values, AUC_test1, col = "green", pch = 18)
legend("topright", legend = c("Test Error", "F1 Score", "AUC"), col = c("blue", "red", "green"), pch = c(16, 17, 18))

# Apply Logistic Regression with SMOTE for Data2
glm2 <- logistic_regression_smote(Data2_train_processed, Data2_test_processed, response_col = "DUMMY_KILLED")
print(glm2)
# $Train_Error_glm
# [1] 0.3645302 0.4371130 0.4972230 0.4999552 0.4999549 0.4999549 0.4999549 0.4999549 0.4999549
# 
# $Test_Error_glm
# [1] 0.327607622 0.055802963 0.005946863 0.001505650 0.001499591 0.001499591 0.001499591
# [8] 0.001499591 0.001499591
# 
# $F1_Score_train
# [1] 0.62127387 0.29191176 0.01945405 0.00000000 0.00000000 0.00000000 0.00000000 0.00000000
# [9] 0.00000000
# 
# $F1_Score_test
# [1] 0.005206704 0.007756949 0.009086320 0.000000000 0.000000000 0.000000000 0.000000000
# [8] 0.000000000 0.000000000
# 
# $AUC_train
# [1] 0.6987129 0.6987129 0.6987129 0.6987129 0.6987129 0.6987129 0.6987129 0.6987129 0.6987129
# 
# $AUC_test
# [1] 0.6658506 0.6658506 0.6658506 0.6658506 0.6658506 0.6658506 0.6658506 0.6658506 0.6658506

# Plot and evaluate errors
Test_Error_glm2 = glm2$Test_Error_glm
F1_Score_test2 = glm2$F1_Score_test
AUC_test2 = glm2$AUC_test

plot(cutoff_values, Test_Error_glm2)

plot(cutoff_values, Test_Error_glm2, type = "b", col = "blue", pch = 16, xlab = "Cutoff", ylab = "Error")
lines(cutoff_values, F1_Score_test2, col = "red", pch = 17)
lines(cutoff_values, AUC_test2, col = "green", pch = 18)
legend("topright", legend = c("Test Error", "F1 Score", "AUC"), col = c("blue", "red", "green"), pch = c(16, 17, 18))


# -------------------- Logistic Regression with Stepwise Selection --------------------
# Logistic Regression with Stepwise Selection and SMOTE for imbalanced data
# Required libraries
library(caret)
library(pROC)

# Cutoff values to evaluate
cutoff_values <- seq(0.1, 0.9, 0.1)

logistic_regression_step_smote <- function(train_data, test_data, response_col = "DUMMY_KILLED") {
  
  # Separate features and target
  X_train <- train_data[, -which(names(train_data) == response_col)]
  y_train <- train_data[[response_col]]
  
  # Apply SMOTE to balance the data
  smote_output <- SMOTE(X_train, y_train, K = 5)
  train_smote <- smote_output$data
  colnames(train_smote)[ncol(train_smote)] <- response_col
  train_smote[[response_col]] <- as.factor(train_smote[[response_col]]) # Fit the logistic regression model using glm (with weights for the imbalance)
  
  # Logistic regression model with stepwise feature selection
  mod_glms_test <- glm(as.factor(DUMMY_KILLED) ~ ., family = binomial, data = train_smote,
                       weights = ifelse(train_smote$DUMMY_KILLED == 1, 1, 10))
  
  # Perform stepwise feature selection using base R's step() function
  mod_glms_test_step <- stats::step(mod_glms_test, direction = "both")
  
  Train_Error_glms <- NULL
  Test_Error_glms <- NULL
  F1_Score_train <- NULL
  F1_Score_test <- NULL
  AUC_train <- NULL
  AUC_test <- NULL
  
  # Calculate Train and Test Errors for different cutoff values
  for (c in cutoff_values) {
    # Train predictions
    train_preds <- predict(mod_glms_test_step, train_smote[, -which(names(train_smote) == response_col)], type = "response")
    y1hat <- ifelse(train_preds < c, 0, 1)
    
    # Test predictions
    test_preds <- predict(mod_glms_test_step, test_data[, -which(names(test_data) == response_col)], type = "response")
    y2hat <- ifelse(test_preds < c, 0, 1)
    
    # Store the error for the current cutoff value
    Train_Error_glms <- c(Train_Error_glms, mean(y1hat != train_smote[[response_col]]))
    Test_Error_glms <- c(Test_Error_glms, mean(y2hat != test_data[[response_col]]))
    
    # Calculate F1 Score for training and testing
    F1_train <- F1_Score(train_smote[[response_col]], y1hat)
    F1_test <- F1_Score(test_data[[response_col]], y2hat)
    F1_Score_train <- c(F1_Score_train, F1_train)
    F1_Score_test <- c(F1_Score_test, F1_test)
    
    # Calculate AUC for training and testing
    train_probs <- predict(mod_glms_test_step, newdata = train_smote[, -which(names(train_smote) == response_col)], type = "response")
    test_probs <- predict(mod_glms_test_step, newdata = test_data[, -which(names(test_data) == response_col)], type = "response")
    
    AUC_train <- c(AUC_train, auc(roc(train_smote[[response_col]], train_probs)))
    AUC_test <- c(AUC_test, auc(roc(test_data[[response_col]], test_probs)))
  }
  
  # Return all metrics as a list
  return(list(Train_Error_glms = Train_Error_glms, 
              Test_Error_glms = Test_Error_glms,
              F1_Score_train = F1_Score_train,
              F1_Score_test = F1_Score_test,
              AUC_train = AUC_train,
              AUC_test = AUC_test))
}

# Apply Logistic Regression with Stepwise Selection and SMOTE for Data1
glms1 <- logistic_regression_step_smote(Data1_train_processed, Data1_test_processed, response_col = "DUMMY_KILLED")
# Start:  AIC=8142916
# as.factor(DUMMY_KILLED) ~ BOROUGH + ZIP_CODE + LATITUDE_CRASH + 
#   LONGITUDE_CRASH + PRCP + SNOW + SNWD + TMAX + TMIN + YEAR + 
#   MONTH + DAY + HOUR + RUSH_HOUR + SEASON + WEEKDAY + GROUPED_FACTOR_VEHICLE_1 + 
#   GROUPED_FACTOR_VEHICLE_2 + VEHICLE_1_TYPE_REGROUP + VEHICLE_2_TYPE_REGROUP
# 
# Df Deviance     AIC
# <none>                         8142874 8142916
# - LONGITUDE_CRASH           1  8142890 8142930
# - ZIP_CODE                  1  8142954 8142994
# - SEASON                    1  8143146 8143186
# - TMAX                      1  8143267 8143307
# - LATITUDE_CRASH            1  8143319 8143359
# - PRCP                      1  8143396 8143436
# - MONTH                     1  8143531 8143571
# - TMIN                      1  8144125 8144165
# - VEHICLE_1_TYPE_REGROUP    1  8144710 8144750
# - DAY                       1  8145109 8145149
# - SNOW                      1  8146239 8146279
# - WEEKDAY                   1  8146341 8146381
# - SNWD                      1  8146842 8146882
# - BOROUGH                   1  8151163 8151203
# - HOUR                      1  8156102 8156142
# - GROUPED_FACTOR_VEHICLE_2  1  8175948 8175988
# - GROUPED_FACTOR_VEHICLE_1  1  8187436 8187476
# - RUSH_HOUR                 1  8201772 8201812
# - YEAR                      1  8298538 8298578
# - VEHICLE_2_TYPE_REGROUP    1  8396844 8396884
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases

print(glms1)
# $Train_Error_glms
# [1] 0.3488310 0.4505211 0.4962516 0.5000140 0.4999549 0.4999549 0.4999549 0.4999549 0.4999549
# 
# $Test_Error_glms
# [1] 0.333857433 0.064182496 0.009124784 0.001645006 0.001499591 0.001499591 0.001499591
# [8] 0.001499591 0.001499591
# 
# $F1_Score_train
# [1] 0.64577495 0.26485027 0.02935613 0.00000000 0.00000000 0.00000000 0.00000000 0.00000000
# [9] 0.00000000
# 
# $F1_Score_test
# [1] 0.005289334 0.008331773 0.008558262 0.000000000 0.000000000 0.000000000 0.000000000
# [8] 0.000000000 0.000000000
# 
# $AUC_train
# [1] 0.7154463 0.7154463 0.7154463 0.7154463 0.7154463 0.7154463 0.7154463 0.7154463 0.7154463
# 
# $AUC_test
# [1] 0.6805972 0.6805972 0.6805972 0.6805972 0.6805972 0.6805972 0.6805972 0.6805972 0.6805972

# Plot and evaluate errors
Test_Error_glms1 = glms1$Test_Error_glms
F1_Score_test1 = glms1$F1_Score_test
AUC_test1 = glms1$AUC_test

plot(cutoff_values, Test_Error_glms1)

plot(cutoff_values, Test_Error_glms1, type = "b", col = "blue", pch = 16, xlab = "Cutoff", ylab = "Error")
lines(cutoff_values, F1_Score_test1, col = "red", pch = 17)
lines(cutoff_values, AUC_test1, col = "green", pch = 18)
legend("topright", legend = c("Test Error", "F1 Score", "AUC"), col = c("blue", "red", "green"), pch = c(16, 17, 18))

# Apply Logistic Regression with Stepwise Selection and SMOTE for Data2
glms2 <- logistic_regression_step_smote(Data2_train_processed, Data2_test_processed, response_col = "DUMMY_KILLED")
# Start:  AIC=8227737
# as.factor(DUMMY_KILLED) ~ BOROUGH + RUSH_HOUR + YEAR + VEHICLE_1_TYPE_REGROUP + 
#   VEHICLE_2_TYPE_REGROUP + LATITUDE_CRASH + GROUPED_FACTOR_VEHICLE_1 + 
#   SNOW + SNWD + PRCP + TMIN
# 
# Df Deviance     AIC
# <none>                         8227713 8227737
# - VEHICLE_1_TYPE_REGROUP    1  8228751 8228773
# - LATITUDE_CRASH            1  8229560 8229582
# - SNOW                      1  8230535 8230557
# - TMIN                      1  8231613 8231635
# - PRCP                      1  8231796 8231818
# - SNWD                      1  8234546 8234568
# - BOROUGH                   1  8240873 8240895
# - GROUPED_FACTOR_VEHICLE_1  1  8276997 8277019
# - RUSH_HOUR                 1  8322961 8322983
# - YEAR                      1  8385996 8386018
# - VEHICLE_2_TYPE_REGROUP    1  8453719 8453741
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases
# Setting levels: control = 0, case = 1
# Setting direction: controls < cases

print(glms2)
# $Train_Error_glms
# [1] 0.3643967 0.4374225 0.4972218 0.4999552 0.4999549 0.4999549 0.4999549 0.4999549 0.4999549
# 
# $Test_Error_glms
# [1] 0.327701536 0.055751462 0.005934745 0.001505650 0.001499591 0.001499591 0.001499591
# [8] 0.001499591 0.001499591
# 
# $F1_Score_train
# [1] 0.62149073 0.29100182 0.01941302 0.00000000 0.00000000 0.00000000 0.00000000 0.00000000
# [9] 0.00000000
# 
# $F1_Score_test
# [1] 0.005205220 0.007764059 0.009104704 0.000000000 0.000000000 0.000000000 0.000000000
# [8] 0.000000000 0.000000000
# 
# $AUC_train
# [1] 0.6986635 0.6986635 0.6986635 0.6986635 0.6986635 0.6986635 0.6986635 0.6986635 0.6986635
# 
# $AUC_test
# [1] 0.6658876 0.6658876 0.6658876 0.6658876 0.6658876 0.6658876 0.6658876 0.6658876 0.6658876

# Plot and evaluate errors
Test_Error_glms2 = glms2$Test_Error_glms
F1_Score_test2 = glms2$F1_Score_test
AUC_test2 = glms2$AUC_test

plot(cutoff_values, Test_Error_glms2)

plot(cutoff_values, Test_Error_glms2, type = "b", col = "blue", pch = 16, xlab = "Cutoff", ylab = "Error")
lines(cutoff_values, F1_Score_test2, col = "red", pch = 17)
lines(cutoff_values, AUC_test2, col = "green", pch = 18)
legend("topright", legend = c("Test Error", "F1 Score", "AUC"), col = c("blue", "red", "green"), pch = c(16, 17, 18))


## ----Monte Carlo Cross-Validation (not implemented as logistic regression seems not a good model for this study--------------------------------
# ------------------------------------------------------------------------------------------

# Set parameters
B <- 100  # Number of loops
set.seed(6242)  # Set the seed for randomization

# Initialize error storage
TRERR1 <- NULL  # Average training errors for dataset 1
TEALL1 <- NULL  # Final test errors for dataset 1
TRERR2 <- NULL  # Average training errors for dataset 2
TEALL2 <- NULL  # Final test errors for dataset 2

for (b in 1:B) {
  
  # Randomly select a set of indices for the test and train split
  flag <- sort(sample(1:n, n1))
  
  # Split data into train and test sets
  Data1_train <- Data1[-flag, ]
  Data1_test <- Data1[flag, ]
  Data2_train <- Data2[-flag, ]
  Data2_test <- Data2[flag, ]
  
  # Define response variables for both datasets
  y1 <- Data1$Response
  y2 <- Data2$Response
  
  # Initialize error vectors
  TrainErr1_CV <- NULL
  TestErr1_CV <- NULL
  TrainErr2_CV <- NULL
  TestErr2_CV <- NULL
  
  # Logistic Regression for Dataset 1
  mod_glm1_CV <- glm(Response ~ ., family = binomial, data = Data1_train)
  c <- 0.5  # Cutoff value
  TrainErr1_CV <- c(TrainErr1_CV, mean(ifelse(predict(mod_glm1_CV, Data1_train[, -c(1, 2)], type = 'response') > c, 1, 0) != y1))
  TestErr1_CV <- c(TestErr1_CV, mean(ifelse(predict(mod_glm1_CV, Data1_test[, -c(1, 2)], type = 'response') > c, 1, 0) != y2))
  
  # Logistic Regression for Dataset 2
  mod_glm2_CV <- glm(Response ~ ., family = binomial, data = Data2_train)
  TrainErr2_CV <- c(TrainErr2_CV, mean(ifelse(predict(mod_glm2_CV, Data2_train[, -c(1, 2)], type = 'response') > c, 1, 0) != y1))
  TestErr2_CV <- c(TestErr2_CV, mean(ifelse(predict(mod_glm2_CV, Data2_test[, -c(1, 2)], type = 'response') > c, 1, 0) != y2))
  
  # Logistic Regression with step() for Dataset 1
  mod_glms1_CV <- step(glm(Response ~ ., family = binomial, data = Data1_train))
  TrainErr1_CV <- c(TrainErr1_CV, mean(ifelse(predict(mod_glms1_CV, Data1_train[, -c(1, 2)], type = 'response') > c, 1, 0) != y1))
  TestErr1_CV <- c(TestErr1_CV, mean(ifelse(predict(mod_glms1_CV, Data1_test[, -c(1, 2)], type = 'response') > c, 1, 0) != y2))
  
  # Logistic Regression with step() for Dataset 2
  mod_glms2_CV <- step(glm(Response ~ ., family = binomial, data = Data2_train))
  TrainErr2_CV <- c(TrainErr2_CV, mean(ifelse(predict(mod_glms2_CV, Data2_train[, -c(1, 2)], type = 'response') > c, 1, 0) != y1))
  TestErr2_CV <- c(TestErr2_CV, mean(ifelse(predict(mod_glms2_CV, Data2_test[, -c(1, 2)], type = 'response') > c, 1, 0) != y2))
  
  # Store results for each iteration
  TRERR1 <- rbind(TRERR1, TrainErr1_CV)
  TEALL1 <- rbind(TEALL1, TestErr1_CV)
  TRERR2 <- rbind(TRERR2, TrainErr2_CV)
  TEALL2 <- rbind(TEALL2, TestErr2_CV)
}

# Results summary
dim(TEALL1)  # 100 5
dim(TEALL2)  # 100 5

colnames(TEALL1) <- c("Logistic Regression", "Logistic Regression 2")
colnames(TEALL2) <- c("Logistic Regression", "Logistic Regression 2")

# CV result statistics
mean_TRERR1 <- apply(TRERR1, 2, mean)
mean_TEALL1 <- apply(TEALL1, 2, mean)
var_TEALL1 <- apply(TEALL1, 2, var)

mean_TRERR2 <- apply(TRERR2, 2, mean)
mean_TEALL2 <- apply(TEALL2, 2, mean)
var_TEALL2 <- apply(TEALL2, 2, var)

# Plot CV Error
k <- c(1, 2, 3, 4, 5)

# Plot for Dataset 1 and Dataset 2
plot(k, mean_TEALL1, xlab = 'Models', ylab = 'CV Error', main = "CV Error Plot: Dataset 1")
plot(k, mean_TEALL2, xlab = 'Models', ylab = 'CV Error', main = "CV Error Plot: Dataset 2")

# Combined plot
TEALL <- cbind(mean_TEALL1, mean_TEALL2)
matplot(k, TEALL, type = "l", lty = 1, col = c("red", "blue"), xlab = "k", ylab = "CV Error", main = "CV Error Plot")
legend("topright", legend = c("Dataset 1", "Dataset 2"), col = c("red", "blue"), lty = 1, cex = 0.5)

## -----------------------------------------------------------------------------
# ## Statistical difference testing on Dataset 1
# T1_1 <- t.test(TEALL1[,1], TEALL1[,2], paired = TRUE)
# T2_1 <- t.test(TEALL1[,1], TEALL1[,3], paired = TRUE)
# T3_1 <- t.test(TEALL1[,1], TEALL1[,4], paired = TRUE)
# T4_1 <- t.test(TEALL1[,1], TEALL1[,5], paired = TRUE)
# 
# W1_1 <- wilcox.test(TEALL1[,1], TEALL1[,2], paired = TRUE)
# W2_1 <- wilcox.test(TEALL1[,1], TEALL1[,3], paired = TRUE)
# W3_1 <- wilcox.test(TEALL1[,1], TEALL1[,4], paired = TRUE)
# W4_1 <- wilcox.test(TEALL1[,1], TEALL1[,5], paired = TRUE)
# 
# p_values_T1_1 <- T1_1$p.value
# p_values_T2_1 <- T2_1$p.value
# p_values_T3_1 <- T3_1$p.value
# p_values_T4_1 <- T4_1$p.value
# 
# p_values_W1_1 <- W1_1$p.value
# p_values_W2_1 <- W2_1$p.value
# p_values_W3_1 <- W3_1$p.value
# p_values_W4_1 <- W4_1$p.value
# 
# # Results
# c(p_values_T1_1, p_values_T2_1, p_values_T3_1, p_values_T4_1)
# c(p_values_W1_1, p_values_W2_1, p_values_W3_1, p_values_W4_1)
# 
# # Boxplot for Dataset 1
# boxplot(TEALL1)
# 
# ## Statistical difference testing on Dataset 2
# T1_2 <- t.test(TEALL2[,1], TEALL2[,2], paired = TRUE)
# T2_2 <- t.test(TEALL2[,1], TEALL2[,3], paired = TRUE)
# T3_2 <- t.test(TEALL2[,1], TEALL2[,4], paired = TRUE)
# T4_2 <- t.test(TEALL2[,1], TEALL2[,5], paired = TRUE)
# 
# W1_2 <- wilcox.test(TEALL2[,1], TEALL2[,2], paired = TRUE)
# W2_2 <- wilcox.test(TEALL2[,1], TEALL2[,3], paired = TRUE)
# W3_2 <- wilcox.test(TEALL2[,1], TEALL2[,4], paired = TRUE)
# W4_2 <- wilcox.test(TEALL2[,1], TEALL2[,5], paired = TRUE)
# 
# p_values_T1_2 <- T1_2$p.value
# p_values_T2_2 <- T2_2$p.value
# p_values_T3_2 <- T3_2$p.value
# p_values_T4_2 <- T4_2$p.value
# 
# p_values_W1_2 <- W1_2$p.value
# p_values_W2_2 <- W2_2$p.value
# p_values_W3_2 <- W3_2$p.value
# p_values_W4_2 <- W4_2$p.value
# 
# # Results
# c(p_values_T1_2, p_values_T2_2, p_values_T3_2, p_values_T4_2)
# c(p_values_W1_2, p_values_W2_2, p_values_W3_2, p_values_W4_2)
# 
# # Boxplot for Dataset 2
# boxplot(TEALL2)

#####################Random Forest###########################################
# --------------------------- Random Forest ---------------------------------
# 1. **Random Forest with SMOTE and Variable Importance**
# ----------------------------------Random Forest with Parameter Tuning-------------------------------------
library(randomForest)
library(caret)
library(e1071)
update.packages("smotefamily")
library(smotefamily) # For SMOTE
library(ROCR)

# 1. Prepare Data and Apply SMOTE from smotefamily
# 10% data
X_train_10 <- Data1_train_10_processed[, -which(names(Data1_train_10_processed) == "DUMMY_KILLED")]
y_train_10 <- Data1_train_10_processed$DUMMY_KILLED
X_test_10 <- Data1_test_10_processed[, -which(names(Data1_test_10_processed) == "DUMMY_KILLED")]
y_test_10 <- Data1_test_10_processed$DUMMY_KILLED

# full data
X_train <- Data1_train_processed[, -which(names(Data1_train_processed) == "DUMMY_KILLED")]
y_train <- Data1_train_processed$DUMMY_KILLED
X_test <- Data1_test_processed[, -which(names(Data1_test_processed) == "DUMMY_KILLED")]
y_test <- Data1_test_processed$DUMMY_KILLED

# Apply SMOTE using smotefamily
# 10% data
smote_output_10 <- SMOTE(X_train_10, y_train_10, K = 5) # K is the number of nearest neighbors
train_smote_10 <- smote_output_10$data
train_smote_10$class <- as.factor(train_smote_10$class) # Ensure it's a factor

smote_output_test_10 <- SMOTE(X_test_10, y_test_10, K = 5) # K is the number of nearest neighbors
test_smote_10 <- smote_output_test_10$data
test_smote_10$class <- as.factor(test_smote_10$class) # Ensure it's a factor

# full data
smote_output <- SMOTE(X_train, y_train, K = 5) # K is the number of nearest neighbors
train_smote <- smote_output$data
train_smote$class <- as.factor(train_smote$class) # Ensure it's a factor
# head(train_smote)
smote_output_test <- SMOTE(X_test, y_test, K = 5) # K is the number of nearest neighbors
test_smote <- smote_output_test$data
test_smote$class <- as.factor(test_smote$class) # Ensure it's a factor

# 2. Set up cross-validation and parameter tuning grid for Random Forest
set.seed(111)

# Define tuning grid for Random Forest
# sqrt(dim(train_smote)[2] -1) #[1] 4.472136
# (dim(train_smote)[2] -1)/3 #[1] 6.666667
tuneGrid <- expand.grid(.mtry = c(4, 5, 6, 7)) # (3,5) for progress report # Number of variables to be randomly sampled at each split
ntrees <- c(500, 1000, 1500)    # c(100, 200, 300)  for progress report
nodesize <- c(1, 5, 10)
param_grid <- expand.grid(ntrees = ntrees,
                          nodesize = nodesize)

# 3. Set up cross-validation with SMOTE resampling
cv_folds <- createFolds(train_smote_10$class, k = 3, returnTrain = TRUE)

ctrl <- trainControl(method = "cv",
                     number = 3,
                     search = 'grid',
                     classProbs = TRUE,
                     savePredictions = "final",
                     index = cv_folds,
                     summaryFunction = twoClassSummary)

# 4. Train Random Forest model with tuning using caret
# rf_tune <- train(
#   make.names(class) ~ .,  # Formula
#   data = train_smote_10,  # Training data
#   method = "rf",  # Random Forest method
#   trControl = ctrl,  # Cross-validation control
#   tuneGrid = tuneGrid,  # Parameter tuning grid
#   importance = TRUE,  # Calculate variable importance
#   metric = "ROC"  # Optimize for ROC AUC
# )

data_maxnode <- vector("list", nrow(param_grid))
for(i in 1:nrow(param_grid)){
  ntree <- param_grid[i,1]
  nodesize <- param_grid[i,2]
  set.seed(333)
  rf_model <- train(make.names(class)~.,
                    data = train_smote_10,
                    method = "rf",
                    importance=TRUE,
                    metric = "ROC",
                    trControl = ctrl,
                    tuneGrid = tuneGrid,
                    ntree = ntree,
                    nodesize = nodesize)
  data_maxnode[[i]] <- rf_model
}

names(data_maxnode) <- paste("ntrees:", param_grid$ntrees,
                             "nodesize:", param_grid$nodesize)

results_mtry <- resamples(data_maxnode)

summary(results_mtry)
# Call:
# summary.resamples(object = results_mtry)
# 
# Models: ntrees: 500 nodesize: 1, ntrees: 1000 nodesize: 1, ntrees: 1500 nodesize: 1, ntrees: 500 nodesize: 5, ntrees: 1000 nodesize: 5, ntrees: 1500 nodesize: 5, ntrees: 500 nodesize: 10, ntrees: 1000 nodesize: 10, ntrees: 1500 nodesize: 10 
# Number of resamples: 3 
# 
# ROC 
# Min.   1st Qu.    Median      Mean   3rd Qu.      Max. NA's
# ntrees: 500 nodesize: 1   0.9999608 0.9999690 0.9999771 0.9999728 0.9999788 0.9999805    0
# ntrees: 1000 nodesize: 1  0.9999657 0.9999698 0.9999740 0.9999735 0.9999774 0.9999808    0
# ntrees: 1500 nodesize: 1  0.9999721 0.9999761 0.9999802 0.9999778 0.9999806 0.9999811    0
# ntrees: 500 nodesize: 5   0.9999704 0.9999751 0.9999798 0.9999789 0.9999832 0.9999865    0
# ntrees: 1000 nodesize: 5  0.9999745 0.9999785 0.9999825 0.9999807 0.9999838 0.9999852    0
# ntrees: 1500 nodesize: 5  0.9999759 0.9999795 0.9999831 0.9999815 0.9999843 0.9999854    0
# ntrees: 500 nodesize: 10  0.9999693 0.9999752 0.9999812 0.9999776 0.9999818 0.9999824    0
# ntrees: 1000 nodesize: 10 0.9999727 0.9999773 0.9999819 0.9999791 0.9999824 0.9999829    0
# ntrees: 1500 nodesize: 10 0.9999769 0.9999799 0.9999828 0.9999820 0.9999845 0.9999861    0
# 
# Sens 
#                           Min. 1st Qu. Median Mean 3rd Qu. Max. NA's
# ntrees: 500 nodesize: 1      1       1      1    1       1    1    0
# ntrees: 1000 nodesize: 1     1       1      1    1       1    1    0
# ntrees: 1500 nodesize: 1     1       1      1    1       1    1    0
# ntrees: 500 nodesize: 5      1       1      1    1       1    1    0
# ntrees: 1000 nodesize: 5     1       1      1    1       1    1    0
# ntrees: 1500 nodesize: 5     1       1      1    1       1    1    0
# ntrees: 500 nodesize: 10     1       1      1    1       1    1    0
# ntrees: 1000 nodesize: 10    1       1      1    1       1    1    0
# ntrees: 1500 nodesize: 10    1       1      1    1       1    1    0
# 
# Spec 
# Min.   1st Qu.    Median      Mean   3rd Qu.      Max. NA's
# ntrees: 500 nodesize: 1   0.9985884 0.9986111 0.9986339 0.9986339 0.9986567 0.9986794    0
# ntrees: 1000 nodesize: 1  0.9985884 0.9985998 0.9986111 0.9986187 0.9986339 0.9986567    0
# ntrees: 1500 nodesize: 1  0.9985884 0.9985884 0.9985884 0.9986263 0.9986453 0.9987022    0
# ntrees: 500 nodesize: 5   0.9984062 0.9984404 0.9984745 0.9984745 0.9985087 0.9985428    0
# ntrees: 1000 nodesize: 5  0.9984062 0.9984518 0.9984973 0.9984821 0.9985201 0.9985428    0
# ntrees: 1500 nodesize: 5  0.9984062 0.9984518 0.9984973 0.9984821 0.9985201 0.9985428    0
# ntrees: 500 nodesize: 10  0.9983379 0.9984176 0.9984973 0.9984442 0.9984973 0.9984973    0
# ntrees: 1000 nodesize: 10 0.9983379 0.9984176 0.9984973 0.9984442 0.9984973 0.9984973    0
# ntrees: 1500 nodesize: 10 0.9983379 0.9984290 0.9985201 0.9984594 0.9985201 0.9985201    0

# to get the best average performance for each model
lapply(data_maxnode, function(x) x$results[x$results$ROC == max(x$results$ROC),])
# $`ntrees: 500 nodesize: 1`
# mtry       ROC Sens      Spec        ROCSD SensSD      SpecSD
# 1    4 0.9999728    1 0.9986339 1.051513e-05      0 4.55363e-05
# 
# $`ntrees: 1000 nodesize: 1`
# mtry       ROC Sens      Spec       ROCSD SensSD       SpecSD
# 1    4 0.9999735    1 0.9986187 7.55521e-06      0 3.477893e-05
# 
# $`ntrees: 1500 nodesize: 1`
# mtry       ROC Sens      Spec        ROCSD SensSD       SpecSD
# 1    4 0.9999778    1 0.9986263 4.960232e-06      0 6.572599e-05
# 
# $`ntrees: 500 nodesize: 5`
# mtry       ROC Sens      Spec        ROCSD SensSD       SpecSD
# 1    4 0.9999789    1 0.9984745 8.061729e-06      0 6.830446e-05
# 
# $`ntrees: 1000 nodesize: 5`
# mtry       ROC Sens      Spec        ROCSD SensSD       SpecSD
# 1    4 0.9999807    1 0.9984821 5.543858e-06      0 6.955785e-05
# 
# $`ntrees: 1500 nodesize: 5`
# mtry       ROC Sens      Spec        ROCSD SensSD       SpecSD
# 1    4 0.9999815    1 0.9984821 4.977966e-06      0 6.955785e-05
# 
# $`ntrees: 500 nodesize: 10`
# mtry       ROC Sens      Spec        ROCSD SensSD       SpecSD
# 1    4 0.9999776    1 0.9984442 7.230966e-06      0 9.201639e-05
# 
# $`ntrees: 1000 nodesize: 10`
# mtry       ROC Sens      Spec        ROCSD SensSD       SpecSD
# 1    4 0.9999791    1 0.9984442 5.604947e-06      0 9.201639e-05
# 
# $`ntrees: 1500 nodesize: 10`
# mtry      ROC Sens      Spec        ROCSD SensSD       SpecSD
# 1    4 0.999982    1 0.9984594 4.675315e-06      0 0.0001051616

# View the best tuning parameters
# print(rf_tune$bestTune)  # Best combination of parameters

# 5. Model summary and performance evaluation
# print(rf_tune)  # Output results of cross-validation

# RF model with tuned parameters
rf_model <- randomForest(as.factor(class) ~., data=train_smote_10, ntree= 1500, 
                         mtry=4, nodesize =10, importance=TRUE)
# Variable Importance
# varImpPlot(rf_tune$finalModel, main = "Variable Importance (Tuned Random Forest)")
varImpPlot(rf_model, main = "Variable Importance (Tuned Random Forest)")

# 6. Prediction on Training Data
rf.pred_train <- predict(rf_model, train_smote_10, type = "prob")[, 2]  # Get probabilities for the positive class
rf.pred_class_train <- predict(rf_model, train_smote_10, type = "class")  # Class predictions for training

# Confusion Matrix and Performance Evaluation for Training Data
conf_matrix_rf_train <- confusionMatrix(as.factor(rf.pred_class_train), as.factor(train_smote$class))
print(conf_matrix_rf_train)
# Confusion Matrix and Statistics

# Reference
# Prediction      0      1
# 0 131868    160
# 1      0 131603
# 
# Accuracy : 0.9994          
# 95% CI : (0.9993, 0.9995)
# No Information Rate : 0.5002          
# P-Value [Acc > NIR] : < 2.2e-16       
# 
# Kappa : 0.9988          
# 
# Mcnemar's Test P-Value : < 2.2e-16       
#                                           
#             Sensitivity : 1.0000          
#             Specificity : 0.9988          
#          Pos Pred Value : 0.9988          
#          Neg Pred Value : 1.0000          
#              Prevalence : 0.5002          
#          Detection Rate : 0.5002          
#    Detection Prevalence : 0.5008          
#       Balanced Accuracy : 0.9994          
#                                           
#        'Positive' Class : 0    

# Training Error (1 - Accuracy)
train_accuracy <- conf_matrix_rf_train$overall['Accuracy']
train_error <- 1 - train_accuracy
cat("Training Error: ", round(train_error, 7), "\n")
# Training Error:  0.0006069

# Training F1 Score
# Extract Precision and Recall from confusion matrix
f1_score_train <- F1_Score(as.factor(rf.pred_class_train), as.factor(train_smote_10$class))
cat("Training F1 Score: ", round(f1_score_train, 7), "\n")
# Training F1 Score:  0.9993937  

# Training AUC
# ROC Curve and AUC for Training Data
prediction_rf_train <- prediction(rf.pred_train, train_smote_10$class)
perf_rf_train <- performance(prediction_rf_train, "tpr", "fpr")
plot(perf_rf_train, main = "ROC Curve (Training Data - Random Forest)")
abline(a = 0, b = 1, lty = 2) # Diagonal line for reference
auc_rf_train <- performance(prediction_rf_train, "auc")@y.values[[1]]
cat("AUC on Training Data (Random Forest):", round(auc_rf_train, 7 ), "\n")
# AUC on Training Data (Random Forest): 1 

# 6. Prediction on Test Data
rf.pred <- predict(rf_model, Data1_test_10_processed, type = "prob")[, 2] # Get probabilities for the positive class
rf.pred_class <- predict(rf_model, Data1_test_10_processed, type = "class")

# Confusion Matrix and Performance Evaluation for Random Forest
conf_matrix_rf <- confusionMatrix(as.factor(rf.pred_class), as.factor(Data1_test_10_processed$DUMMY_KILLED))
print(conf_matrix_rf)
# Confusion Matrix and Statistics
# 
# Reference
# Prediction     0     1
# 0 32966    43
# 1     0     0
# 
# Accuracy : 0.9987          
# 95% CI : (0.9982, 0.9991)
# No Information Rate : 0.9987          
# P-Value [Acc > NIR] : 0.5404          
# 
# Kappa : 0               
# 
# Mcnemar's Test P-Value : 1.504e-10       
#                                           
#             Sensitivity : 1.0000          
#             Specificity : 0.0000          
#          Pos Pred Value : 0.9987          
#          Neg Pred Value :    NaN          
#              Prevalence : 0.9987          
#          Detection Rate : 0.9987          
#    Detection Prevalence : 1.0000          
#       Balanced Accuracy : 0.5000          
#                                           
#        'Positive' Class : 0        

# 1-0.9987 # [1] 0.0013

# Testing Error (1 - Accuracy)
test_accuracy <- conf_matrix_rf$overall['Accuracy']
test_error <- 1 - test_accuracy
cat("Testing Error: ", round(test_error, 7), "\n")
# Testing Error:  0.0013027 

# Testing F1 Score
f1_score_test <- F1_Score(as.factor(rf.pred_class), as.factor(Data1_test_10_processed$DUMMY_KILLED))
cat("Testing F1 Score: ", round(f1_score_test, 7), "\n")
# Testing F1 Score:  0

# ROC Curve and AUC for Random Forest
prediction_rf <- prediction(rf.pred, Data1_test_10_processed$DUMMY_KILLED)
perf_rf <- performance(prediction_rf, "tpr", "fpr")
plot(perf_rf, main = "ROC Curve (Tuned Random Forest)")
abline(a = 0, b = 1, lty = 2) # Diagonal line for reference
auc_rf <- performance(prediction_rf, "auc")@y.values[[1]]
cat("AUC on Test Data (Tuned Random Forest):", round(auc_rf, 7), "\n")
# AUC on Test Data (Tuned Random Forest): 0.6323185

##--------------------F1 score as metrics-----------------------
# Define custom trainControl using F1 score
ctrl_F1 <- trainControl(method = "cv",
                        number = 3,
                        search = 'grid',
                        classProbs = TRUE,
                        savePredictions = "final",
                        index = cv_folds,
                        summaryFunction = function(data, lev = NULL, model = NULL) {
                          predicted <- data$pred
                          actual <- data$obs
                          f1 <- F1_Score(predicted, actual)
                          return(c(F1 = f1))
                        },
                        verboseIter = TRUE)

# Random Forest Parameter Tuning
data_maxnode_F1 <- vector("list", nrow(param_grid))
for(i in 1:nrow(param_grid)){
  ntree <- param_grid[i,1]
  nodesize <- param_grid[i,2]
  set.seed(333)
  
  # Train Random Forest model with F1 score as the metric
  rf_model_F1 <- train(make.names(class) ~ ., 
                       data = train_smote_10,
                       method = "rf",
                       importance = TRUE,
                       metric = "F1",  # Specify F1 as the evaluation metric
                       trControl = ctrl_F1,
                       tuneGrid = tuneGrid,
                       ntree = ntree,
                       nodesize = nodesize)
  
  # Store the model
  data_maxnode_F1[[i]] <- rf_model_F1
}

# Assign model names based on parameters
names(data_maxnode_F1) <- paste("ntrees:", param_grid$ntrees,
                                "nodesize:", param_grid$nodesize)

# 5. Evaluate results
results_mtry_F1 <- resamples(data_maxnode_F1)
summary(results_mtry_F1)
# Call:
#   summary.resamples(object = results_mtry_F1)
# 
# Models: ntrees: 500 nodesize: 1, ntrees: 1000 nodesize: 1, ntrees: 1500 nodesize: 1, ntrees: 500 nodesize: 5, ntrees: 1000 nodesize: 5, ntrees: 1500 nodesize: 5, ntrees: 500 nodesize: 10, ntrees: 1000 nodesize: 10, ntrees: 1500 nodesize: 10 
# Number of resamples: 3 
# 
# F1 
# Min. 1st Qu. Median Mean 3rd Qu. Max. NA's
# ntrees: 500 nodesize: 1      0       0      0    0       0    0    0
# ntrees: 1000 nodesize: 1     0       0      0    0       0    0    0
# ntrees: 1500 nodesize: 1     0       0      0    0       0    0    0
# ntrees: 500 nodesize: 5      0       0      0    0       0    0    0
# ntrees: 1000 nodesize: 5     0       0      0    0       0    0    0
# ntrees: 1500 nodesize: 5     0       0      0    0       0    0    0
# ntrees: 500 nodesize: 10     0       0      0    0       0    0    0
# ntrees: 1000 nodesize: 10    0       0      0    0       0    0    0
# ntrees: 1500 nodesize: 10    0       0      0    0       0    0    0


#####################Boosting (GBM Model)###########################################
# --------------------------- Boosting (GBM Model) ---------------------------------
# Define parameter grid for tuning
gbmGrid <- expand.grid(
  n.trees = c(50, 100, 500, 1500), 
  interaction.depth = c(1, 3, 5), 
  shrinkage = c(0.01, 0.1), 
  n.minobsinnode = c(1, 5, 10)
)

fitControl <- trainControl(
  method = "cv",
  number = 3,
  classProbs = TRUE,
  sampling = "smote", # Apply SMOTE resampling
  summaryFunction = twoClassSummary # Important for AUC
)

set.seed(222)
# Fit Boosting model using caret
gbmFit <- train(
  make.names(class) ~ ., 
  data = train_smote, 
  method = "gbm", 
  trControl = fitControl, 
  tuneGrid = gbmGrid, 
  metric = "ROC", 
  verbose = FALSE
)


# Print Boosting results
print(gbmFit)
plot(gbmFit)

# 10% data------------------------------------------------------------------
# Stochastic Gradient Boosting 
#
# 263631 samples
# 20 predictor
# 2 classes: 'X0', 'X1' 
# 
# No pre-processing
# Resampling: Cross-Validated (3 fold) 
# Summary of sample sizes: 175754, 175754, 175754 
# Addtional sampling using SMOTE
# 
# Resampling results across tuning parameters:
#   
#   shrinkage  interaction.depth  n.minobsinnode  n.trees  ROC        Sens       Spec     
# 0.01       1                   1                50     0.7415257  0.5348000  0.9439752
# 0.01       1                   1               100     0.8439995  0.5348000  0.9439752
# 0.01       1                   1               500     0.9309174  0.7727804  0.9104453
# 0.01       1                   1              1500     0.9718863  0.8967225  0.9176248
# 0.01       1                   5                50     0.7415239  0.5348000  0.9439752
# 0.01       1                   5               100     0.8518927  0.5348000  0.9439752
# 0.01       1                   5               500     0.9306763  0.7700807  0.9154011
# 0.01       1                   5              1500     0.9717964  0.8957973  0.9174123
# 0.01       1                  10                50     0.7415240  0.5348000  0.9439752
# 0.01       1                  10               100     0.8439982  0.5348000  0.9439752
# 0.01       1                  10               500     0.9315986  0.7734780  0.9092537
# 0.01       1                  10              1500     0.9718701  0.8959869  0.9169190
# 0.01       3                   1                50     0.8781851  0.7625883  0.8270683
# 0.01       3                   1               100     0.9353807  0.7637410  0.8833815
# 0.01       3                   1               500     0.9916315  0.9670656  0.9514355
# 0.01       3                   1              1500     0.9992509  0.9955410  0.9784158
# 0.01       3                   5                50     0.8781878  0.7625883  0.8270683
# 0.01       3                   5               100     0.9361530  0.7644083  0.8833891
# 0.01       3                   5               500     0.9916166  0.9670276  0.9489234
# 0.01       3                   5              1500     0.9992367  0.9953211  0.9784689
# 0.01       3                  10                50     0.8781847  0.7625883  0.8270607
# 0.01       3                  10               100     0.9348896  0.7627628  0.8833891
# 0.01       3                  10               500     0.9915831  0.9669973  0.9502516
# 0.01       3                  10              1500     0.9992571  0.9955562  0.9782336
# 0.01       5                   1                50     0.9155036  0.9212015  0.7878843
# 0.01       5                   1               100     0.9525629  0.9094018  0.8777806
# 0.01       5                   1               500     0.9979455  0.9885416  0.9726327
# 0.01       5                   1              1500     0.9996774  0.9996891  0.9972375
# 0.01       5                   5                50     0.9175721  0.9212015  0.7878843
# 0.01       5                   5               100     0.9527052  0.9104028  0.8748207
# 0.01       5                   5               500     0.9979169  0.9883823  0.9725796
# 0.01       5                   5              1500     0.9996795  0.9996967  0.9969870
# 0.01       5                  10                50     0.9173073  0.9212015  0.7878767
# 0.01       5                  10               100     0.9524020  0.9116389  0.8718988
# 0.01       5                  10               500     0.9979153  0.9885264  0.9725644
# 0.01       5                  10              1500     0.9996800  0.9997118  0.9971236
# 0.10       1                   1                50     0.9301264  0.7850047  0.9080243
# 0.10       1                   1               100     0.9595319  0.8472184  0.9096180
# 0.10       1                   1               500     0.9914117  0.9703719  0.9544561
# 0.10       1                   1              1500     0.9990748  0.9951542  0.9785372
# 0.10       1                   5                50     0.9300011  0.7885234  0.9027420
# 0.10       1                   5               100     0.9596549  0.8493569  0.9098609
# 0.10       1                   5               500     0.9913984  0.9705008  0.9540387
# 0.10       1                   5              1500     0.9990731  0.9951391  0.9782185
# 0.10       1                  10                50     0.9300504  0.7827524  0.9062863
# 0.10       1                  10               100     0.9597827  0.8497740  0.9078042
# 0.10       1                  10               500     0.9913758  0.9701975  0.9538262
# 0.10       1                  10              1500     0.9990764  0.9951012  0.9782868
# 0.10       3                   1                50     0.9917001  0.9677936  0.9477774
# 0.10       3                   1               100     0.9979001  0.9882989  0.9685496
# 0.10       3                   1               500     0.9996782  0.9999621  0.9986567
# 0.10       3                   1              1500     0.9983642  0.9984909  0.9986794
# 0.10       3                   5                50     0.9911434  0.9676571  0.9458725
# 0.10       3                   5               100     0.9979052  0.9878212  0.9695059
# 0.10       3                   5               500     0.9996993  0.9999848  0.9986643
# 0.10       3                   5              1500     0.9994288  0.9994919  0.9985352
# 0.10       3                  10                50     0.9913103  0.9672096  0.9473145
# 0.10       3                  10               100     0.9979173  0.9881397  0.9695059
# 0.10       3                  10               500     0.9996906  0.9999848  0.9986643
# 0.10       3                  10              1500     0.9990970  0.9994919  0.9982848
# 0.10       5                   1                50     0.9979742  0.9890345  0.9745148
# 0.10       5                   1               100     0.9996156  0.9988397  0.9913481
# 0.10       5                   1               500     0.9995360  0.9995905  0.9986794
# 0.10       5                   1              1500     0.9988570  0.9987867  0.9986794
# 0.10       5                   5                50     0.9978298  0.9884202  0.9732095
# 0.10       5                   5               100     0.9996077  0.9985137  0.9931847
# 0.10       5                   5               500     0.9996967  0.9996512  0.9986719
# 0.10       5                   5              1500     0.9972834  0.9973307  0.9985352
# 0.10       5                  10                50     0.9978364  0.9879501  0.9711603
# 0.10       5                  10               100     0.9996240  0.9985212  0.9928204
# 0.10       5                  10               500     0.9997257  0.9999242  0.9986719
# 0.10       5                  10              1500     0.9984837  0.9992947  0.9978370
# 
# ROC was used to select the optimal model using the largest value.
# The final values used for the model were n.trees = 500, interaction.depth = 5, shrinkage = 0.1 and n.minobsinnode = 10.

# 20250407 full data------------------------------------------------------
# Stochastic Gradient Boosting 
# 
# 2636618 samples
# 20 predictor
# 2 classes: 'X0', 'X1' 
# 
# No pre-processing
# Resampling: Cross-Validated (3 fold) 
# Summary of sample sizes: 1757745, 1757746, 1757745 
# Addtional sampling using SMOTE
# 
# Resampling results across tuning parameters:
#   
#   shrinkage  interaction.depth  n.minobsinnode  n.trees  ROC        Sens       Spec     
# 0.01       1                   1                50     0.7903345  0.6944012  0.7676306
# 0.01       1                   1               100     0.8196196  0.6944012  0.7676306
# 0.01       1                   1               500     0.9109475  0.8138397  0.8448433
# 0.01       1                   1              1500     0.9553743  0.8896003  0.8805400
# 0.01       1                   5                50     0.7903344  0.6944012  0.7676306
# 0.01       1                   5               100     0.8196195  0.6944012  0.7676306
# 0.01       1                   5               500     0.9107344  0.8144684  0.8431766
# 0.01       1                   5              1500     0.9553897  0.8896125  0.8809906
# 0.01       1                  10                50     0.7903344  0.6944012  0.7676306
# 0.01       1                  10               100     0.8196196  0.6944012  0.7676306
# 0.01       1                  10               500     0.9109891  0.8136288  0.8448797
# 0.01       1                  10              1500     0.9554391  0.8892332  0.8814033
# 0.01       3                   1                50     0.8931503  0.8686944  0.7188835
# 0.01       3                   1               100     0.9123804  0.8792577  0.7641334
# 0.01       3                   1               500     0.9856326  0.9680271  0.9100433
# 0.01       3                   1              1500     0.9981117  0.9956850  0.9656286
# 0.01       3                   5                50     0.8966303  0.8633570  0.7212640
# 0.01       3                   5               100     0.9130654  0.8792577  0.7615814
# 0.01       3                   5               500     0.9856643  0.9680923  0.9098977
# 0.01       3                   5              1500     0.9981062  0.9956668  0.9655695
# 0.01       3                  10                50     0.8910389  0.8633570  0.7212640
# 0.01       3                  10               100     0.9131720  0.8792577  0.7562324
# 0.01       3                  10               500     0.9856204  0.9679482  0.9100904
# 0.01       3                  10              1500     0.9981328  0.9957882  0.9657986
# 0.01       5                   1                50     0.9114226  0.9225555  0.6975072
# 0.01       5                   1               100     0.9423540  0.9289639  0.8176742
# 0.01       5                   1               500     0.9960606  0.9894003  0.9474105
# 0.01       5                   1              1500     0.9992873  0.9996488  0.9837072
# 0.01       5                   5                50     0.9111877  0.9225555  0.6975064
# 0.01       5                   5               100     0.9430179  0.9275455  0.8201519
# 0.01       5                   5               500     0.9960932  0.9895269  0.9477989
# 0.01       5                   5              1500     0.9992854  0.9996648  0.9836222
# 0.01       5                  10                50     0.9117883  0.9208815  0.7011576
# 0.01       5                  10               100     0.9431338  0.9265459  0.8205426
# 0.01       5                  10               500     0.9960654  0.9893714  0.9474431
# 0.01       5                  10              1500     0.9992848  0.9996663  0.9835631
# 0.10       1                   1                50     0.9119486  0.8146763  0.8428193
# 0.10       1                   1               100     0.9433233  0.8595024  0.8771262
# 0.10       1                   1               500     0.9828262  0.9544662  0.9177220
# 0.10       1                   1              1500     0.9967321  0.9907708  0.9606445
# 0.10       1                   5                50     0.9119948  0.8146459  0.8428095
# 0.10       1                   5               100     0.9432423  0.8598232  0.8770822
# 0.10       1                   5               500     0.9828970  0.9546847  0.9181666
# 0.10       1                   5              1500     0.9967071  0.9907534  0.9604685
# 0.10       1                  10                50     0.9119855  0.8146763  0.8427829
# 0.10       1                  10               100     0.9432782  0.8597997  0.8771118
# 0.10       1                  10               500     0.9827974  0.9543745  0.9176856
# 0.10       1                  10              1500     0.9967051  0.9906282  0.9606127
# 0.10       3                   1                50     0.9855894  0.9689206  0.9091732
# 0.10       3                   1               100     0.9946302  0.9849032  0.9433701
# 0.10       3                   1               500     0.9995763  0.9999628  0.9928288
# 0.10       3                   1              1500     0.9996316  0.9999826  0.9985101
# 0.10       3                   5                50     0.9851963  0.9669584  0.9088773
# 0.10       3                   5               100     0.9951089  0.9862154  0.9458121
# 0.10       3                   5               500     0.9995813  0.9999750  0.9927006
# 0.10       3                   5              1500     0.9996383  0.9999742  0.9985192
# 0.10       3                  10                50     0.9857763  0.9686725  0.9089759
# 0.10       3                  10               100     0.9952606  0.9868965  0.9462020
# 0.10       3                  10               500     0.9995668  0.9999719  0.9926566
# 0.10       3                  10              1500     0.9996375  0.9999970  0.9985192
# 0.10       5                   1                50     0.9959790  0.9893426  0.9476418
# 0.10       5                   1               100     0.9988684  0.9984208  0.9747578
# 0.10       5                   1               500     0.9996209  0.9999886  0.9976384
# 0.10       5                   1              1500     0.9995019  0.9996807  0.9985351
# 0.10       5                   5                50     0.9956124  0.9883005  0.9456778
# 0.10       5                   5               100     0.9988575  0.9983495  0.9736616
# 0.10       5                   5               500     0.9996244  0.9999977  0.9974328
# 0.10       5                   5              1500     0.9995793  0.9998529  0.9985207
# 0.10       5                  10                50     0.9960520  0.9896741  0.9476350
# 0.10       5                  10               100     0.9988746  0.9984664  0.9747017
# 0.10       5                  10               500     0.9996282  0.9999954  0.9974670
# 0.10       5                  10              1500     0.9996196  0.9998271  0.9985253
# 
# ROC was used to select the optimal model using the largest value.
# The final values used for the model were n.trees = 1500, interaction.depth = 3, shrinkage = 0.1 and n.minobsinnode = 5.

# optimal tune values
gbmFit$finalModel$tuneValue
# 10% data------------------------------------------------
# n.trees interaction.depth shrinkage n.minobsinnode
# 71     500                 5       0.1             10

# 20250407 full data--------------------------------------
# n.trees interaction.depth shrinkage n.minobsinnode
# 56    1500                 3       0.1              5

whichTwoPct <- tolerance(gbmFit$results, metric = "ROC", 
                         tol = 2, maximize = TRUE)  
cat("best model within 2 pct of best:")
gbmFit$results[whichTwoPct,1:6]
# 10% data------------------------------------------------
# shrinkage interaction.depth n.minobsinnode n.trees       ROC      Sens
# 49       0.1                 3              1      50 0.9917001 0.9677936

# 20250407 full data------------------------------------------------------
# shrinkage interaction.depth n.minobsinnode n.trees       ROC      Sens
# 49       0.1                 3              1      50 0.9855894 0.9689206

# optimal model
optimal_params <- expand.grid(
  n.trees = 50,               # Optimal n.trees
  interaction.depth = 3,      # Optimal interaction depth
  shrinkage = 0.1,            # Optimal shrinkage
  n.minobsinnode = 1         # Optimal n.minobsinnode
)

# Set up cross-validation parameters (same as before)
fitControl_gbm <- trainControl(
  method = "cv",
  number = 3,                # 3-fold cross-validation
  classProbs = TRUE,         # For calculating probabilities (AUC)
  sampling = "smote",        # SMOTE for class imbalance
  summaryFunction = twoClassSummary, # Use AUC as metric
  savePredictions = "final" # Save final predictions
)

# Set random seed for reproducibility
set.seed(222)

# Train the model using optimal parameters
gbmFit_optimal <- train(
  make.names(class) ~ ., 
  data = train_smote, 
  method = "gbm", 
  trControl = fitControl_gbm, 
  tuneGrid = optimal_params, 
  metric = "ROC", 
  verbose = FALSE
)

# Check model details
print(gbmFit_optimal)
# 10% data------------------------------------------------
# Stochastic Gradient Boosting 
# 
# 263631 samples
# 20 predictor
# 2 classes: 'X0', 'X1' 
# 
# No pre-processing
# Resampling: Cross-Validated (3 fold) 
# Summary of sample sizes: 175754, 175754, 175754 
# Addtional sampling using SMOTE
# 
# Resampling results:
#   
#   ROC        Sens       Spec     
# 0.9911565  0.9651621  0.9440283
# 
# Tuning parameter 'n.trees' was held constant at a value of 50
# Tuning parameter 'interaction.depth' was held constant
# at a value of 3
# Tuning parameter 'shrinkage' was held constant at a value of 0.1
# Tuning parameter 'n.minobsinnode'
# was held constant at a value of 1

# 20250407 full data------------------------------------------------------
# Stochastic Gradient Boosting 
# 
# 2636618 samples
# 20 predictor
# 2 classes: 'X0', 'X1' 
# 
# No pre-processing
# Resampling: Cross-Validated (3 fold) 
# Summary of sample sizes: 1757745, 1757746, 1757745 
# Addtional sampling using SMOTE
# 
# Resampling results:
#   
#   ROC        Sens       Spec     
# 0.9853021  0.9685148  0.9080133
# 
# Tuning parameter 'n.trees' was held constant at a value of 50
# Tuning parameter 'interaction.depth' was held constant
# at a value of 3
# Tuning parameter 'shrinkage' was held constant at a value of 0.1
# Tuning parameter 'n.minobsinnode'
# was held constant at a value of 1

# Predict on the training data
train_predictions <- predict(gbmFit_optimal, newdata = train_smote)
# unique(train_predictions)

train_predictions <- factor(train_predictions, levels = c("X0", "X1"), labels = c(0, 1))
# unique(train_predictions)

# Confusion matrix for training data
train_cm <- confusionMatrix(train_predictions, train_smote$class)
print(train_cm)
# 10% data------------------------------------------------
# Confusion Matrix and Statistics
# 
# Reference
# Prediction      0      1
# 0 128065   6930
# 1   3803 124833

# Accuracy : 0.9593        
# 95% CI : (0.9585, 0.96)
# No Information Rate : 0.5002        
# P-Value [Acc > NIR] : < 2.2e-16     
# 
# Kappa : 0.9186        
# 
# Mcnemar's Test P-Value : < 2.2e-16     
#                                         
#             Sensitivity : 0.9712        
#             Specificity : 0.9474        
#          Pos Pred Value : 0.9487        
#          Neg Pred Value : 0.9704        
#              Prevalence : 0.5002        
#          Detection Rate : 0.4858        
#    Detection Prevalence : 0.5121        
#       Balanced Accuracy : 0.9593        
#                                         
#        'Positive' Class : 0      

# 1-0.9593 # 0.0407

# 20250407 full data------------------------------------------------------
# Confusion Matrix and Statistics
# 
# Reference
# Prediction       0       1
# 0 1275491  119061
# 1   42937 1199129
# 
# Accuracy : 0.9386          
# 95% CI : (0.9383, 0.9388)
# No Information Rate : 0.5             
# P-Value [Acc > NIR] : < 2.2e-16       
# 
# Kappa : 0.8771          
# 
# Mcnemar's Test P-Value : < 2.2e-16       
#                                           
#             Sensitivity : 0.9674          
#             Specificity : 0.9097          
#          Pos Pred Value : 0.9146          
#          Neg Pred Value : 0.9654          
#              Prevalence : 0.5000          
#          Detection Rate : 0.4838          
#    Detection Prevalence : 0.5289          
#       Balanced Accuracy : 0.9386          
#                                           
#        'Positive' Class : 0     

# Training Error (1 - Accuracy)
train_accuracy <- train_cm$overall['Accuracy']
train_error <- 1 - train_accuracy
cat("Training Error: ", round(train_error, 7), "\n")
# 20250407 full data----------------------------------------
# Training Error:  0.0614416 

# AUC for training data
train_probs <- predict(gbmFit_optimal, newdata = train_smote, type = "prob")
train_auc <- roc(train_smote$class, train_probs[, 2]) # AUC for positive class
print(paste("Training AUC: ", auc(train_auc)))
# 10% data------------------------------------------------
# [1] "Training AUC:  0.992066780729949"
# 20250407 full data------------------------------------------------------
# [1] "Training AUC:  0.985139079546631"

# Calculate F1-Score for training data
f1_train <- F1_Score(train_smote$class, train_predictions)
print(paste("Training F1-Score: ", f1_train))
# [1] "Training F1-Score:  0.958782483803701"
# 20250407 full data------------------------------------------------------
# [1] "Training F1-Score:  0.936725858664133

# Predict on the test data
test_predictions <- predict(gbmFit_optimal, newdata = Data1_test_processed)
test_predictions <- factor(test_predictions, levels = c("X0", "X1"), labels = c(0, 1))

# unique(test_predictions)
# unique(as.factor(Data1_test_10_processed$DUMMY_KILLED))

# Confusion matrix for test data
test_cm <- confusionMatrix(test_predictions, as.factor(Data1_test_processed$DUMMY_KILLED))
print(test_cm)
# 10% data------------------------------------------------
# Reference
# Prediction     0     1
# 0 32060    39
# 1   906     4

# Reference for test_smote
# Prediction     0     1
# 0 32060  3361
# 1   906 29577

# 20250407 full data------------------------------------------------------
# Confusion Matrix and Statistics
# 
# Reference
# Prediction      0      1
# 0 318756    441
# 1  10839     54
# 
# Accuracy : 0.9658          
# 95% CI : (0.9652, 0.9664)
# No Information Rate : 0.9985          
# P-Value [Acc > NIR] : 1               
# 
# Kappa : 0.0066          
# 
# Mcnemar's Test P-Value : <2e-16          
#                                           
#             Sensitivity : 0.967114        
#             Specificity : 0.109091        
#          Pos Pred Value : 0.998618        
#          Neg Pred Value : 0.004957        
#              Prevalence : 0.998500        
#          Detection Rate : 0.965664        
#    Detection Prevalence : 0.967000        
#       Balanced Accuracy : 0.538103        
#                                           
#        'Positive' Class : 0    

# Testing Error (1 - Accuracy)
test_accuracy <- test_cm$overall['Accuracy']
test_error <- 1 - test_accuracy
cat("Testing Error: ", round(test_error, 7), "\n")
# 20250407 full data------------------------------------------------------
# Testing Error:  0.0341725

# AUC for test data
test_probs <- predict(gbmFit_optimal, newdata = Data1_test_processed, type = "prob")
test_auc <- roc(Data1_test_processed$DUMMY_KILLED, test_probs[, 2]) # AUC for positive class
print(paste("Test AUC: ", auc(test_auc)))
# 10% data------------------------------------------------
# [1] "Test AUC:  0.64294713792505"
# [1] "Test AUC:  0.987939824874243" for test_smote
# 20250407 full data------------------------------------------------------
# [1] "Test AUC:  0.64294713792505"

# Calculate F1-Score for test data
f1_test <- F1_Score(Data1_test_processed$DUMMY_KILLED, test_predictions)
print(paste("Test F1-Score: ", f1_test))
# 10% data------------------------------------------------
# [1] "Test F1-Score:  0.00839454354669465" 
# [1] "Test F1-Score:  0.932719446240204" for test_smote
# 20250407 full data------------------------------------------------------
# [1] "Test F1-Score:  0.00948366701791359"

# # ROC Curve and AUC for Boosting
# prediction_gbm <- prediction(gbm_pred, Data1_test_processed$DUMMY_KILLED)
# perf_gbm <- performance(prediction_gbm, "tpr", "fpr")
# plot(perf_gbm, main = "ROC Curve (Boosting)")
# abline(a = 0, b = 1, lty = 2) # Diagonal line for reference
# auc_gbm <- performance(prediction_gbm, "auc")@y.values[[1]]
# cat("AUC on Test Data (Boosting):", round(auc_gbm, 4), "\n")
# # AUC on Test Data (Boosting): 0.6813 

# 3. **Variable Importance for Boosting**
# Variable importance from the trained GBM model
importance_gbm <- varImp(gbmFit_optimal, scale = FALSE)
print(importance_gbm)
# 10% data------------------------------------------------
# gbm variable importance
# 
# Overall
# GROUPED_FACTOR_VEHICLE_1 37076.4
# RUSH_HOUR                29218.4
# VEHICLE_2_TYPE_REGROUP   23456.8
# VEHICLE_1_TYPE_REGROUP   11428.3
# YEAR                      8486.7
# WEEKDAY                   6860.0
# SNWD                      4922.4
# SEASON                    3960.7
# HOUR                      3816.2
# SNOW                       469.4
# MONTH                      434.7
# PRCP                       414.1
# TMIN                       327.7
# LATITUDE_CRASH               0.0
# LONGITUDE_CRASH              0.0
# GROUPED_FACTOR_VEHICLE_2     0.0
# BOROUGH                      0.0
# DAY                          0.0
# TMAX                         0.0
# ZIP_CODE                     0.0

# 20250407 full data------------------------------------------------------
# gbm variable importance
# 
# Overall
# RUSH_HOUR                 332764
# GROUPED_FACTOR_VEHICLE_1  298865
# VEHICLE_2_TYPE_REGROUP    197354
# VEHICLE_1_TYPE_REGROUP    135126
# YEAR                       97955
# WEEKDAY                    91763
# SNWD                       30704
# HOUR                       28098
# PRCP                        4968
# SNOW                        4001
# GROUPED_FACTOR_VEHICLE_2    1403
# DAY                            0
# SEASON                         0
# LATITUDE_CRASH                 0
# MONTH                          0
# BOROUGH                        0
# TMAX                           0
# LONGITUDE_CRASH                0
# ZIP_CODE                       0
# TMIN                           0

# Plot variable importance for Boosting
plot(importance_gbm, main = "Variable Importance (Boosting)")

## Model Inspection 
## Find the estimated optimal number of iterations
gbm_model <- gbmFit_optimal$finalModel
perf_gbm = gbm.perf(gbm_model) 
# OOB generally underestimates the optimal number of iterations although predictive performance is reasonably competitive. Using cv_folds>1 when calling gbm usually results in improved predictive performance.
perf_gbm 
# 10% data------------------------------------------------
# [1] 50
# attr(,"smoother")
# Call:
#   loess(formula = object$oobag.improve ~ x, enp.target = min(max(4, 
#                                                                  length(x)/10), 50))
# 
# Number of Observations: 50 
# Equivalent Number of Parameters: 4.48 
# Residual Standard Error: 0.002396 

# 20250407 full data------------------------------------------------------
# [1] 50
# attr(,"smoother")
# Call:
#   loess(formula = object$oobag.improve ~ x, enp.target = min(max(4, 
#                                                                  length(x)/10), 50))
# 
# Number of Observations: 50 
# Equivalent Number of Parameters: 4.48 
# Residual Standard Error: 0.00203 

# Add predictions to dataframe (Cor_Data)
cor_predictions <- predict(gbmFit_optimal, newdata = Cor_Data)
cor_predictions_numeric <- factor(cor_predictions, levels = c("X0", "X1"), labels = c(0, 1))
cor_predictions_numeric <- as.numeric(as.character(cor_predictions_numeric))
Cor_Data$PREDICTED_DUMMY_KILLED <- cor_predictions_numeric
# head(Cor_Data)
cm <- confusionMatrix(as.factor(cor_predictions_numeric), as.factor(Cor_Data$DUMMY_KILLED))
print(cm)
# 10% data------------------------------------------------
# Confusion Matrix and Statistics

# Reference
# Prediction       0       1
# 0 1587236    2165
# 1   60787     260
# 
# Accuracy : 0.9619          
# 95% CI : (0.9616, 0.9621)
# No Information Rate : 0.9985          
# P-Value [Acc > NIR] : 1               
# 
# Kappa : 0.0054          
# 
# Mcnemar's Test P-Value : <2e-16          
#                                           
#             Sensitivity : 0.963115        
#             Specificity : 0.107216        
#          Pos Pred Value : 0.998638        
#          Neg Pred Value : 0.004259        
#              Prevalence : 0.998531        
#          Detection Rate : 0.961700        
#    Detection Prevalence : 0.963012        
#       Balanced Accuracy : 0.535166        
#                                           
#        'Positive' Class : 0    

# 20250407 full data------------------------------------------------------
# Confusion Matrix and Statistics
# 
# Reference
# Prediction       0       1
# 0 1594247    2131
# 1   53776     294
# 
# Accuracy : 0.9661          
# 95% CI : (0.9658, 0.9664)
# No Information Rate : 0.9985          
# P-Value [Acc > NIR] : 1               
# 
# Kappa : 0.0076          
# 
# Mcnemar's Test P-Value : <2e-16          
#                                           
#             Sensitivity : 0.967369        
#             Specificity : 0.121237        
#          Pos Pred Value : 0.998665        
#          Neg Pred Value : 0.005437        
#              Prevalence : 0.998531        
#          Detection Rate : 0.965948        
#    Detection Prevalence : 0.967239        
#       Balanced Accuracy : 0.544303        
#                                           
#        'Positive' Class : 0   

# Save to CSV
write.csv(Cor_Data, "data_with_predictions.csv", row.names = FALSE)

